lets do llm call and prompt first

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import httpx

#llm call
llm_connection = ChatOpenAI(api_key=os.environ['OPENAI_API_KEY'],
                            base_url="https://alertmanager.tecnicasreunidas.es/v1",
                            http_client=httpx.Client(verify=False),
                            model = "gpt-5-mini",
                            temperature=0)

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a {tone} assistant"),
    ("user","Generate answer for the users requested information : {topic}.")
])

user_set_tone = input("Enter the tone you want to set for the llm...")
user_input = input("Enter the topic you like to ask the llm..")

final_prompt = prompt_template.invoke({'tone':user_set_tone,"topic":user_input})

llm_answer = llm_connection.invoke(final_prompt.messages).content
llm_answer

'Sure — here’s a compact, useful overview of Telugu books and where to find them. If you tell me what you specifically want (classics, modern fiction, poetry, children’s books, religious texts, ebooks, translations, or learning Telugu), I can give a tailored list.\n\nQuick categories & recommended reads (representative, not exhaustive)\n- Classical/epics\n  - Andhra Mahabharatam — the Telugu rendering by Nannaya, Tikkana and Errana\n  - Pothana — Andhra Maha Bhagavatamu\n\n- Modern classics & social drama\n  - Gurajada Apparao — Kanyasulkam (a landmark social play)\n  - Viswanatha Satyanarayana — Veyi Padagalu (Veyipadagalu) (major Telugu novel)\n\n- Progressive/modern poetry\n  - Srirangam Srinivasa Rao (Sri Sri) — Maha Prasthanam (modern, revolutionary poetry)\n  - Gurram Jashuva — major progressive poet\n\n- 20th-century writers & novelists\n  - C. Narayana Reddy — notable poet/novelist (Viswambhara is a famous work)\n  - Chalam (Gudipati Venkatachalam) — modernist novelist tackling

now lets get the output in schema that is key value pair!

In [2]:
llm_results = llm_connection.invoke("Tell me 2 weired factts in key value pair format with fact1 , fact2 as keys and information as value")
llm_results.content

'fact1: Honey never spoils — archaeologists have found pots of honey in ancient Egyptian tombs thousands of years old that are still edible due to honey’s low water content and natural preservatives.  \nfact2: Bananas are botanically berries, while strawberries are not — strawberries are aggregate accessory fruits formed from many ovaries of a single flower.'

# using pydantic model 

In [3]:
from pydantic import BaseModel

class llm_schema(BaseModel):
    setup : str
    punchline : str

In [4]:
obj = llm_schema(**{"setup":"somesetup","punchline":"punchline"})
obj.setup

'somesetup'

In [ ]:

llm_structured_output = llm_connection.with_structured_output(llm_schema)
result = llm_structured_output.invoke("tell me a science joke")

result.punchline

'Because they make up everything.'

In [8]:
result.setup

"Why can't you trust atoms?"

# *it is better to define the description in the pydantic class for better llm understanding

In [10]:
from pydantic import BaseModel,Field

class llm_schema(BaseModel):
    setup : str = Field(description = "The setup for the joke")
    punchline : str = Field(description = "The puncline for the joke")

In [11]:
llm_structured_output = llm_connection.with_structured_output(llm_schema)
llm_structured_output.invoke("Tell me a joke")

llm_schema(setup="Why don't programmers like nature?", punchline='It has too many bugs.')

# *now using typeddict*
it is like pydantic .. but simple

In [12]:
from typing import TypedDict

class llm_schema_td(TypedDict):
    setup : str
    punchline : str

In [15]:
obj = llm_schema_td({"setup":"some setup", "punchline":"some Punch line"})
obj #here obj is a simple dict


{'setup': 'some setup', 'punchline': 'some Punch line'}

In [17]:
llm_structured_td = llm_connection.with_structured_output(llm_schema_td)
result = llm_structured_td.invoke("tell me a joke")
result

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field.'}

In [1]:
 from langchain_openai import OpenAI